# NC-Vibration: Bearing Fault Detection on MCU
**Noise-Conditioned SSM/TCN for CWRU Bearing Dataset**

3 Blue Oceans:
1. MCU + Mamba = **0 prior work**
2. <10K params + noise robustness = **0 prior work**
3. Leakage-free + systematic SNR eval = **almost none**

| Model | Params | INT8 | Target MCU |
|-------|--------|------|------------|
| NC-Vib-TCN-Tiny | 2,929 | 2.9 KB | $0.5 M0+ (per-motor) |
| NC-Vib-TCN-Matched | 5,309 | 5.2 KB | $2 M4 |
| NC-Vib-SSM-20K | 19,630 | 19.2 KB | $7 M7 |
| NC-Vib-TCN-20K | 21,385 | 20.9 KB | $7 M7 |

In [ ]:
# 1. Clone repo
import os

dst = '/content/NanoMamba-Interspeech2026'
if not os.path.exists(dst):
    !git clone https://github.com/DrJinHoChoi/NC-SSM-TASLP2026.git {dst}
    print('Clone done!')
else:
    os.chdir(dst)
    !git pull
    print('Updated!')

os.chdir(dst)
print(f'Working dir: {os.getcwd()}')

In [ ]:
!pip install -q torchaudio scipy

In [ ]:
# 2. Download CWRU dataset (~40 .mat files, ~50MB total)
!python train_vibration.py --download --data_dir ./data/cwru

In [ ]:
# 3. Model verification + speed comparison
import sys
sys.path.insert(0, '/content/NanoMamba-Interspeech2026')

from nc_vibration import *
import torch, time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}\n')

models = {
    'NC-Vib-TCN-Tiny': create_nc_vib_tcn_tiny(),
    'NC-Vib-TCN-Matched': create_nc_vib_tcn_matched(),
    'NC-Vib-SSM-Matched': create_nc_vib_ssm_matched(),
    'NC-Vib-TCN-20K': create_nc_vib_tcn_20k(),
    'NC-Vib-SSM-20K': create_nc_vib_ssm_20k(),
}

vib = torch.randn(1, 2048, device=device)  # 0.17s @ 12kHz

print(f'{"Model":<24} {"Params":>8} {"INT8":>8} {"Latency":>10}')
print('-' * 54)

for name, m in models.items():
    m = m.to(device).eval()
    p = sum(pp.numel() for pp in m.parameters())
    
    # Warmup
    with torch.no_grad():
        for _ in range(10): m(vib)
    if device == 'cuda': torch.cuda.synchronize()
    
    # Benchmark
    t0 = time.perf_counter()
    for _ in range(50):
        with torch.no_grad(): m(vib)
    if device == 'cuda': torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / 50 * 1000
    
    print(f'{name:<24} {p:>8,} {p/1024:>6.1f}KB {ms:>8.1f}ms')

In [ ]:
# 4. Train: Leakage-free cross-load protocol
#    Train: 0,1,2 HP  ->  Test: 3 HP (unseen load condition)
!python train_vibration.py \
    --models "NC-Vib-TCN-Tiny,NC-Vib-TCN-Matched,NC-Vib-TCN-20K,NC-Vib-SSM-20K" \
    --protocol load_split \
    --train_loads 0,1,2 --test_loads 3 \
    --epochs 50 \
    --batch_size 64 \
    --lr 1e-3 \
    --noise_aug \
    --n_classes 4 \
    --data_dir ./data/cwru \
    --checkpoint_dir ./checkpoints_vibration

In [ ]:
# 5. Noise robustness evaluation (4 noise types x 6 SNR levels)
!python train_vibration.py \
    --models "NC-Vib-TCN-Tiny,NC-Vib-TCN-20K,NC-Vib-SSM-20K" \
    --eval_only --noise_eval \
    --protocol load_split \
    --n_classes 4 \
    --data_dir ./data/cwru \
    --checkpoint_dir ./checkpoints_vibration

In [ ]:
# 6. 10-class (fault severity) experiment
!python train_vibration.py \
    --models "NC-Vib-TCN-20K,NC-Vib-SSM-20K" \
    --protocol load_split \
    --epochs 50 --batch_size 64 --lr 1e-3 \
    --noise_aug --n_classes 10 \
    --data_dir ./data/cwru \
    --checkpoint_dir ./checkpoints_vibration_10class

In [ ]:
# 7. Severity-split protocol: Train on 0.014/0.021 -> Test on 0.007 (unseen severity)
!python train_vibration.py \
    --models "NC-Vib-TCN-20K" \
    --protocol severity_split \
    --epochs 50 --batch_size 64 --lr 1e-3 \
    --noise_aug --n_classes 4 \
    --data_dir ./data/cwru \
    --checkpoint_dir ./checkpoints_vibration_severity

In [ ]:
# 8. Save best checkpoints to Drive
from google.colab import drive
import shutil

drive.mount('/content/drive')

for src_dir in ['./checkpoints_vibration', './checkpoints_vibration_10class']:
    if os.path.exists(src_dir):
        dst_dir = f'/content/drive/MyDrive/NC-SSM-checkpoints/vibration/{os.path.basename(src_dir)}'
        os.makedirs(os.path.dirname(dst_dir), exist_ok=True)
        shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
        print(f'Saved: {dst_dir}')
    else:
        print(f'Not found: {src_dir} - train first')